In [ ]:
import os
import re
import gc
import random

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)

from arabert.preprocess import ArabertPreprocessor


SEED = 42

PERTURBATION_SEEDS = [
    42,
    43,
    44,
    45,
    46
]

SEVERITIES = [
    0.10,
    0.20,
    0.30
]

DIMENSIONS = [
    "Textual Accuracy",
    "Completeness",
    "Consistency",
    "Validity",
    "Understandability"
]


DATA_PATH = "ArSAS.txt"

TEXT_COLUMN = "Tweet_text"

LABEL_COLUMN = "Sentiment_label"


DROP_MIXED_CLASS = False


REMOVE_CONFLICTING_DUPLICATES = True


DEDUPLICATE_IDENTICAL_TEXTS = True


MAX_LENGTH = 128

LEARNING_RATE = 2e-5

TRAIN_BATCH_SIZE = 16

EVAL_BATCH_SIZE = 32

NUM_EPOCHS = 3

WEIGHT_DECAY = 0.01


OUTPUT_DIR = (
    "./ArSAS_Sentiment_5_dimensions"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


MODELS = {

    "AraBERTv2": {

        "model_name":
            "aubmindlab/bert-base-arabertv2",

        "preprocessing":
            "arabert"
    },


    "CAMeLBERT-Mix": {

        "model_name":
            "CAMeL-Lab/bert-base-arabic-camelbert-mix",

        "preprocessing":
            "raw"
    },


    "XLM-R": {

        "model_name":
            "FacebookAI/xlm-roberta-base",

        "preprocessing":
            "raw"
    }
}


def set_all_seeds(seed):

    set_seed(seed)

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)


set_all_seeds(SEED)


print(
    "CUDA available:",
    torch.cuda.is_available()
)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


df = pd.read_csv(

    DATA_PATH,

    sep="\t"
)


print(
    "\nOriginal dataset shape:",
    df.shape
)


print(
    "\nColumns:"
)

print(
    df.columns.tolist()
)


required_columns = {

    TEXT_COLUMN,

    LABEL_COLUMN
}


missing_columns = (

    required_columns

    -

    set(df.columns)
)


if missing_columns:

    raise ValueError(

        f"Missing required columns: "
        f"{missing_columns}"
    )


df = (

    df[
        [
            TEXT_COLUMN,
            LABEL_COLUMN
        ]
    ]

    .copy()

    .rename(
        columns={

            TEXT_COLUMN:
                "text",

            LABEL_COLUMN:
                "labels"
        }
    )
)


df = (

    df

    .dropna(
        subset=[
            "text",
            "labels"
        ]
    )

    .reset_index(drop=True)
)


df["text"] = (

    df["text"]

    .astype(str)

    .str.strip()
)


df["labels"] = (

    df["labels"]

    .astype(str)

    .str.strip()

    .str.title()
)


df = (

    df[
        df["text"] != ""
    ]

    .reset_index(drop=True)
)


print(
    "\nDataset after basic cleaning:",
    df.shape
)


print(
    "\nOriginal sentiment distribution:"
)

print(
    df[
        "labels"
    ]
    .value_counts()
)


EXPECTED_CLASSES = {

    "Negative",

    "Neutral",

    "Positive",

    "Mixed"
}


observed_classes = set(

    df[
        "labels"
    ].unique()
)


unexpected_classes = (

    observed_classes

    -

    EXPECTED_CLASSES
)


if unexpected_classes:

    raise ValueError(

        "Unexpected sentiment classes found: "
        f"{unexpected_classes}"
    )


if DROP_MIXED_CLASS:

    print(
        "\nWARNING:"
        " Mixed class is being removed."
    )


    df = (

        df[
            df["labels"]
            !=
            "Mixed"
        ]

        .reset_index(drop=True)
    )


label_counts_per_text = (

    df

    .groupby(
        "text"
    )[
        "labels"
    ]

    .nunique()
)


conflicting_texts = set(

    label_counts_per_text[

        label_counts_per_text
        >
        1

    ].index
)


print(
    "\nTexts with conflicting sentiment labels:",
    len(conflicting_texts)
)


duplicate_rows = int(

    df[
        "text"
    ]

    .duplicated()

    .sum()
)


print(
    "Duplicate text rows:",
    duplicate_rows
)


if (
    REMOVE_CONFLICTING_DUPLICATES
    and
    conflicting_texts
):

    df = (

        df[
            ~df["text"]
            .isin(
                conflicting_texts
            )
        ]

        .reset_index(drop=True)
    )


    print(
        "\nRemoved all examples having "
        "conflicting sentiment labels."
    )


if DEDUPLICATE_IDENTICAL_TEXTS:

    before = len(df)


    df = (

        df

        .drop_duplicates(
            subset=[
                "text"
            ],

            keep="first"
        )

        .reset_index(drop=True)
    )


    removed = (
        before
        -
        len(df)
    )


    print(
        "Identical duplicate rows removed:",
        removed
    )


print(
    "\nFinal clean dataset shape:",
    df.shape
)


print(
    "\nFinal sentiment distribution:"
)

print(
    df[
        "labels"
    ]

    .value_counts()
)


label_encoder = (
    LabelEncoder()
)


df[
    "labels"
] = (

    label_encoder

    .fit_transform(
        df[
            "labels"
        ]
    )
)


num_labels = len(
    label_encoder.classes_
)


id2label = {

    idx:
        str(label)

    for idx, label

    in enumerate(
        label_encoder.classes_
    )
}


label2id = {

    str(label):
        idx

    for idx, label

    in enumerate(
        label_encoder.classes_
    )
}


print(
    "\nNumber of classes:",
    num_labels
)


print(
    "\nLabel mapping:"
)


for idx, label in (
    id2label.items()
):

    print(
        idx,
        "->",
        label
    )


train_df, test_df = (

    train_test_split(

        df,

        test_size=0.20,

        random_state=SEED,

        stratify=df[
            "labels"
        ]
    )
)


train_df = (

    train_df

    .reset_index(
        drop=True
    )
)


test_df = (

    test_df

    .reset_index(
        drop=True
    )
)


print(
    "\nTrain shape:",
    train_df.shape
)


print(
    "Test shape:",
    test_df.shape
)


train_texts = set(
    train_df[
        "text"
    ]
)


test_texts = set(
    test_df[
        "text"
    ]
)


overlap = (

    train_texts

    .intersection(
        test_texts
    )
)


print(
    "\nExact train-test text overlap:",
    len(overlap)
)


assert (
    len(overlap)
    ==
    0
), (
    "ERROR: Exact text leakage "
    "between train and test."
)


train_df.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "fixed_train_split.csv"
    ),

    index=False
)


test_df.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "fixed_test_split.csv"
    ),

    index=False
)


pd.DataFrame({

    "Label_ID":
        list(
            id2label.keys()
        ),

    "Sentiment":
        list(
            id2label.values()
        )

}).to_csv(

    os.path.join(

        OUTPUT_DIR,

        "label_mapping.csv"
    ),

    index=False
)


def stochastic_count(
    target,
    rng
):

    base = int(

        np.floor(
            target
        )
    )


    fraction = (

        target
        -
        base
    )


    if (
        rng.random()
        <
        fraction
    ):

        base += 1


    return base


ARABIC_PATTERN = re.compile(

    r"[\u0621-\u063A"
    r"\u0641-\u064A"
    r"\u0671-\u06D3"
    r"\u06FA-\u06FC]+"
)


def get_arabic_span(

    token,

    min_len=3

):

    if not isinstance(
        token,
        str
    ):

        return None


    matches = list(

        ARABIC_PATTERN.finditer(
            token
        )
    )


    eligible = [

        match

        for match in matches

        if len(
            match.group()
        ) >= min_len
    ]


    if not eligible:

        return None


    return max(

        eligible,

        key=lambda match:
            len(
                match.group()
            )
    )


ARABIC_CHARS = list(

    "ابتثجحخدذرزسشصضطظعغفقكلمنهوي"
    "أإآؤئءىة"
)


ORTHOGRAPHIC_ALTERNATIVES = {

    "ا": {
        "أ",
        "إ",
        "آ"
    },

    "أ": {
        "ا",
        "إ",
        "آ"
    },

    "إ": {
        "ا",
        "أ",
        "آ"
    },

    "آ": {
        "ا",
        "أ",
        "إ"
    },

    "ي": {
        "ى"
    },

    "ى": {
        "ي"
    }
}


def delete_char(
    word,
    rng
):

    if len(word) < 2:

        return word


    idx = rng.randrange(
        len(word)
    )


    return (

        word[:idx]

        +

        word[
            idx + 1:
        ]
    )


def insert_char(
    word,
    rng
):

    idx = rng.randrange(
        len(word) + 1
    )


    char = rng.choice(
        ARABIC_CHARS
    )


    return (

        word[:idx]

        +

        char

        +

        word[idx:]
    )


def substitute_char(
    word,
    rng
):

    idx = rng.randrange(
        len(word)
    )


    original = (
        word[idx]
    )


    forbidden = {
        original
    }


    forbidden.update(

        ORTHOGRAPHIC_ALTERNATIVES.get(

            original,

            set()
        )
    )


    candidates = [

        char

        for char
        in ARABIC_CHARS

        if char
        not in forbidden
    ]


    if not candidates:

        return word


    replacement = (
        rng.choice(
            candidates
        )
    )


    return (

        word[:idx]

        +

        replacement

        +

        word[
            idx + 1:
        ]
    )


def transpose_chars(
    word,
    rng
):

    valid_positions = [

        idx

        for idx
        in range(
            len(word) - 1
        )

        if (
            word[idx]
            !=
            word[idx + 1]
        )
    ]


    if not valid_positions:

        return word


    idx = rng.choice(
        valid_positions
    )


    chars = list(
        word
    )


    chars[idx], chars[idx + 1] = (

        chars[idx + 1],

        chars[idx]
    )


    return "".join(
        chars
    )


TYPO_OPERATIONS = [

    delete_char,

    insert_char,

    substitute_char,

    transpose_chars
]


def corrupt_word(
    word,
    rng
):

    for _ in range(20):

        operation = (
            rng.choice(
                TYPO_OPERATIONS
            )
        )


        corrupted = operation(
            word,
            rng
        )


        if (
            corrupted
            !=
            word
        ):

            return corrupted


    return insert_char(
        word,
        rng
    )


def perturb_accuracy(

    text,

    severity,

    rng

):

    tokens = (
        text.split()
    )


    eligible = [

        idx

        for idx, token

        in enumerate(
            tokens
        )

        if (
            get_arabic_span(
                token
            )
            is not None
        )
    ]


    n = len(
        eligible
    )


    if n == 0:

        return (
            text,
            0,
            0
        )


    k = stochastic_count(

        severity
        *
        n,

        rng
    )


    k = min(
        k,
        n
    )


    if k == 0:

        return (
            text,
            n,
            0
        )


    selected = (
        rng.sample(
            eligible,
            k
        )
    )


    changed = 0


    for idx in selected:

        token = (
            tokens[idx]
        )


        match = (
            get_arabic_span(
                token
            )
        )


        original_word = (
            match.group()
        )


        corrupted_word = (
            corrupt_word(
                original_word,
                rng
            )
        )


        tokens[idx] = (

            token[
                :match.start()
            ]

            +

            corrupted_word

            +

            token[
                match.end():
            ]
        )


        if (
            corrupted_word
            !=
            original_word
        ):

            changed += 1


    return (

        " ".join(
            tokens
        ),

        n,

        changed
    )


def perturb_completeness(

    text,

    severity,

    rng

):

    words = (
        text.split()
    )


    n = len(
        words
    )


    if n <= 1:

        return (
            text,
            n,
            0
        )


    k = stochastic_count(

        severity
        *
        n,

        rng
    )


    k = min(

        k,

        n - 1
    )


    if k == 0:

        return (
            text,
            n,
            0
        )


    selected = set(

        rng.sample(
            range(n),
            k
        )
    )


    remaining = [

        word

        for idx, word

        in enumerate(
            words
        )

        if idx
        not in selected
    ]


    return (

        " ".join(
            remaining
        ),

        n,

        k
    )


CONSISTENCY_MAP = {

    "ا": [
        "أ",
        "إ",
        "آ"
    ],

    "أ": [
        "ا",
        "إ",
        "آ"
    ],

    "إ": [
        "ا",
        "أ",
        "آ"
    ],

    "آ": [
        "ا",
        "أ",
        "إ"
    ],

    "ي": [
        "ى"
    ],

    "ى": [
        "ي"
    ]
}


def perturb_consistency(

    text,

    severity,

    rng

):

    chars = list(
        text
    )


    eligible = [

        idx

        for idx, char

        in enumerate(
            chars
        )

        if char
        in CONSISTENCY_MAP
    ]


    n = len(
        eligible
    )


    if n == 0:

        return (
            text,
            0,
            0
        )


    k = stochastic_count(

        severity
        *
        n,

        rng
    )


    k = min(
        k,
        n
    )


    if k == 0:

        return (
            text,
            n,
            0
        )


    selected = rng.sample(

        eligible,

        k
    )


    for idx in selected:

        chars[idx] = (
            rng.choice(

                CONSISTENCY_MAP[
                    chars[idx]
                ]
            )
        )


    return (

        "".join(
            chars
        ),

        n,

        k
    )


INVALID_TOKENS = [

    "zxqv999",

    "qvzx777",

    "xqvz555",

    "vzqx333"
]


def perturb_validity(

    text,

    severity,

    rng

):

    words = (
        text.split()
    )


    n = len(
        words
    )


    if n == 0:

        return (
            text,
            0,
            0
        )


    k = stochastic_count(

        severity
        *
        n,

        rng
    )


    if k == 0:

        return (
            text,
            n,
            0
        )


    output = (
        words.copy()
    )


    for _ in range(k):

        invalid_token = (
            rng.choice(
                INVALID_TOKENS
            )
        )


        position = (
            rng.randrange(
                len(output)
                +
                1
            )
        )


        output.insert(

            position,

            invalid_token
        )


    return (

        " ".join(
            output
        ),

        n,

        k
    )


def understandability_count(

    n,

    severity,

    rng

):

    if n < 2:

        return 0


    target = min(

        severity
        *
        n,

        n
    )


    if target < 2:

        probability = (
            target
            /
            2
        )


        if (
            rng.random()
            <
            probability
        ):

            return 2


        return 0


    return min(

        stochastic_count(
            target,
            rng
        ),

        n
    )


def perturb_understandability(

    text,

    severity,

    rng

):

    words = (
        text.split()
    )


    n = len(
        words
    )


    if n < 2:

        return (
            text,
            n,
            0
        )


    k = understandability_count(

        n,

        severity,

        rng
    )


    if k < 2:

        return (
            text,
            n,
            0
        )


    selected = None


    for _ in range(30):

        candidate = sorted(

            rng.sample(
                range(n),
                k
            )
        )


        candidate_words = [

            words[idx]

            for idx
            in candidate
        ]


        if (
            len(
                set(
                    candidate_words
                )
            )
            >
            1
        ):

            selected = candidate

            break


    if selected is None:

        return (
            text,
            n,
            0
        )


    original_words = [

        words[idx]

        for idx
        in selected
    ]


    permuted_words = None


    for _ in range(50):

        candidate = (
            original_words.copy()
        )


        rng.shuffle(
            candidate
        )


        if (
            candidate
            !=
            original_words
        ):

            permuted_words = candidate

            break


    if permuted_words is None:

        return (
            text,
            n,
            0
        )


    output = (
        words.copy()
    )


    for idx, new_word in zip(

        selected,

        permuted_words

    ):

        output[idx] = (
            new_word
        )


    perturbed_text = (
        " ".join(
            output
        )
    )


    actual_changed = sum(

        output[idx]
        !=
        words[idx]

        for idx
        in selected
    )


    if (
        perturbed_text
        ==
        text
    ):

        return (
            text,
            n,
            0
        )


    return (

        perturbed_text,

        n,

        actual_changed
    )


PERTURBATION_FUNCTIONS = {

    "Textual Accuracy":
        perturb_accuracy,

    "Completeness":
        perturb_completeness,

    "Consistency":
        perturb_consistency,

    "Validity":
        perturb_validity,

    "Understandability":
        perturb_understandability
}


def create_perturbed_dataframe(

    clean_df,

    dimension,

    severity,

    seed

):

    rng = (
        random.Random(
            seed
        )
    )


    function = (

        PERTURBATION_FUNCTIONS[
            dimension
        ]
    )


    perturbed_df = (
        clean_df.copy()
    )


    perturbed_texts = []

    total_eligible_units = 0

    total_changed_units = 0

    changed_instances = 0


    for text in (
        clean_df[
            "text"
        ]
    ):

        (
            perturbed_text,
            eligible_units,
            changed_units

        ) = function(

            text,

            severity,

            rng
        )


        perturbed_texts.append(
            perturbed_text
        )


        total_eligible_units += (
            eligible_units
        )


        total_changed_units += (
            changed_units
        )


        if (
            perturbed_text
            !=
            text
        ):

            changed_instances += 1


    perturbed_df[
        "text"
    ] = (
        perturbed_texts
    )


    assert (

        perturbed_df[
            "labels"
        ]

        .equals(

            clean_df[
                "labels"
            ]
        )

    ), (
        "ERROR: Gold labels changed."
    )


    realized_rate = (

        total_changed_units
        /
        total_eligible_units

        if total_eligible_units > 0

        else 0.0
    )


    changed_instance_rate = (

        changed_instances
        /
        len(clean_df)

        if len(clean_df) > 0

        else 0.0
    )


    diagnostics = {

        "Dimension":
            dimension,

        "Severity":
            severity,

        "Seed":
            seed,

        "Eligible_Units":
            total_eligible_units,

        "Changed_Units":
            total_changed_units,

        "Realized_Rate":
            realized_rate,

        "Changed_Instances":
            changed_instances,

        "Changed_Instance_Rate":
            changed_instance_rate
    }


    return (
        perturbed_df,
        diagnostics
    )


print(
    "\n"
    +
    "=" * 100
)


print(
    "GENERATING ALL RAW PERTURBATION CONDITIONS"
)


print(
    "=" * 100
)


PERTURBED_TEST_SETS = {}

ALL_DIAGNOSTICS = []


for dimension in DIMENSIONS:

    for severity in SEVERITIES:

        for perturb_seed in (
            PERTURBATION_SEEDS
        ):

            (
                perturbed_df,
                diagnostics

            ) = (
                create_perturbed_dataframe(

                    clean_df=
                        test_df,

                    dimension=
                        dimension,

                    severity=
                        severity,

                    seed=
                        perturb_seed
                )
            )


            key = (

                dimension,

                severity,

                perturb_seed
            )


            PERTURBED_TEST_SETS[
                key
            ] = (
                perturbed_df
            )


            ALL_DIAGNOSTICS.append(
                diagnostics
            )


diagnostics_df = pd.DataFrame(
    ALL_DIAGNOSTICS
)


diagnostic_summary = (

    diagnostics_df

    .groupby(

        [
            "Dimension",
            "Severity"
        ],

        as_index=False
    )

    .agg(

        Realized_Rate_Mean=(

            "Realized_Rate",

            "mean"
        ),

        Realized_Rate_SD=(

            "Realized_Rate",

            "std"
        ),

        Changed_Instance_Rate_Mean=(

            "Changed_Instance_Rate",

            "mean"
        )
    )
)


diagnostic_display = (
    diagnostic_summary.copy()
)


diagnostic_display[
    "Nominal_Severity"
] = (

    diagnostic_display[
        "Severity"
    ]

    *
    100
)


diagnostic_display[
    "Realized_Rate_Mean"
] *= 100


diagnostic_display[
    "Realized_Rate_SD"
] *= 100


diagnostic_display[
    "Changed_Instance_Rate_Mean"
] *= 100


print(
    "\n"
    +
    "=" * 100
)


print(
    "PERTURBATION CHECK BEFORE TRAINING"
)


print(
    "=" * 100
)


display(

    diagnostic_display[
        [
            "Dimension",

            "Nominal_Severity",

            "Realized_Rate_Mean",

            "Realized_Rate_SD",

            "Changed_Instance_Rate_Mean"
        ]
    ]

    .round(3)
)


def evaluate_dataset(

    trainer,

    dataset

):

    output = (
        trainer.predict(
            dataset
        )
    )


    predictions = np.argmax(

        output.predictions,

        axis=-1
    )


    labels = (
        output.label_ids
    )


    accuracy = (
        accuracy_score(

            labels,

            predictions
        )
    )


    macro_f1 = (
        f1_score(

            labels,

            predictions,

            average="macro",

            zero_division=0
        )
    )


    weighted_f1 = (
        f1_score(

            labels,

            predictions,

            average="weighted",

            zero_division=0
        )
    )


    return {

        "accuracy":
            accuracy,

        "macro_f1":
            macro_f1,

        "weighted_f1":
            weighted_f1,

        "labels":
            labels,

        "predictions":
            predictions
    }


ALL_RESULTS = []

CLEAN_RESULTS = []


for (
    MODEL_LABEL,
    CONFIG
) in MODELS.items():


    print(
        "\n\n"
        +
        "#" * 100
    )


    print(
        "STARTING MODEL:",
        MODEL_LABEL
    )


    print(
        "#" * 100
    )


    set_all_seeds(
        SEED
    )


    MODEL_NAME = (
        CONFIG[
            "model_name"
        ]
    )


    preprocessing_type = (

        CONFIG[
            "preprocessing"
        ]
    )


    if (
        preprocessing_type
        ==
        "arabert"
    ):

        print(
            "\nUsing AraBERT preprocessing."
        )


        arabert_preprocessor = (

            ArabertPreprocessor(

                model_name=
                    MODEL_NAME,

                keep_emojis=
                    True
            )
        )


        def model_preprocess(
            text
        ):

            return (

                arabert_preprocessor

                .preprocess(
                    str(text)
                )
            )


    else:

        print(
            f"\nUsing raw text pipeline "
            f"for {MODEL_LABEL}."
        )


        def model_preprocess(
            text
        ):

            return str(
                text
            )


    print(
        "\nPreparing clean train/test inputs..."
    )


    train_model_df = (
        train_df.copy()
    )


    clean_test_model_df = (
        test_df.copy()
    )


    train_model_df[
        "text"
    ] = (

        train_model_df[
            "text"
        ]

        .apply(
            model_preprocess
        )
    )


    clean_test_model_df[
        "text"
    ] = (

        clean_test_model_df[
            "text"
        ]

        .apply(
            model_preprocess
        )
    )


    tokenizer = (

        AutoTokenizer

        .from_pretrained(
            MODEL_NAME
        )
    )


    model = (

        AutoModelForSequenceClassification

        .from_pretrained(

            MODEL_NAME,

            num_labels=
                num_labels,

            id2label=
                id2label,

            label2id=
                label2id
        )
    )


    def tokenize_function(
        batch
    ):

        return tokenizer(

            batch[
                "text"
            ],

            truncation=True,

            max_length=
                MAX_LENGTH
        )


    train_dataset = (

        Dataset

        .from_pandas(

            train_model_df,

            preserve_index=False
        )

        .map(

            tokenize_function,

            batched=True
        )
    )


    clean_test_dataset = (

        Dataset

        .from_pandas(

            clean_test_model_df,

            preserve_index=False
        )

        .map(

            tokenize_function,

            batched=True
        )
    )


    data_collator = (

        DataCollatorWithPadding(
            tokenizer=tokenizer
        )
    )


    model_output_dir = (
        os.path.join(

            OUTPUT_DIR,

            MODEL_LABEL.replace(
                " ",
                "_"
            )
        )
    )


    training_args = (
        TrainingArguments(

            output_dir=
                model_output_dir,

            learning_rate=
                LEARNING_RATE,

            per_device_train_batch_size=
                TRAIN_BATCH_SIZE,

            per_device_eval_batch_size=
                EVAL_BATCH_SIZE,

            num_train_epochs=
                NUM_EPOCHS,

            weight_decay=
                WEIGHT_DECAY,

            logging_strategy=
                "epoch",

            save_strategy=
                "no",

            report_to=
                "none",

            seed=
                SEED,

            data_seed=
                SEED,

            optim=
                "adamw_torch"
        )
    )


    trainer = Trainer(

        model=model,

        args=training_args,

        train_dataset=
            train_dataset,

        data_collator=
            data_collator
    )


    print(
        "\n========================================"
    )


    print(
        "TRAINING",
        MODEL_LABEL
    )


    print(
        "========================================"
    )


    trainer.train()


    clean_result = (
        evaluate_dataset(

            trainer,

            clean_test_dataset
        )
    )


    clean_accuracy = (
        clean_result[
            "accuracy"
        ]
    )


    clean_macro_f1 = (
        clean_result[
            "macro_f1"
        ]
    )


    clean_weighted_f1 = (
        clean_result[
            "weighted_f1"
        ]
    )


    CLEAN_RESULTS.append({

        "Model":
            MODEL_LABEL,

        "Clean_Accuracy":
            clean_accuracy,

        "Clean_Macro_F1":
            clean_macro_f1,

        "Clean_Weighted_F1":
            clean_weighted_f1
    })


    print(
        "\n========================================"
    )


    print(
        f"{MODEL_LABEL} CLEAN RESULTS"
    )


    print(
        "========================================"
    )


    print(

        f"Accuracy    : "
        f"{clean_accuracy * 100:.2f}"
    )


    print(

        f"Macro-F1    : "
        f"{clean_macro_f1 * 100:.2f}"
    )


    print(

        f"Weighted-F1 : "
        f"{clean_weighted_f1 * 100:.2f}"
    )


    clean_report = (
        classification_report(

            clean_result[
                "labels"
            ],

            clean_result[
                "predictions"
            ],

            labels=list(
                range(
                    num_labels
                )
            ),

            target_names=[

                id2label[idx]

                for idx in range(
                    num_labels
                )
            ],

            output_dict=True,

            zero_division=0
        )
    )


    clean_report_df = (

        pd.DataFrame(
            clean_report
        )

        .transpose()
    )


    clean_report_df.to_csv(

        os.path.join(

            OUTPUT_DIR,

            MODEL_LABEL.replace(
                " ",
                "_"
            )

            +

            "_clean_classification_report.csv"
        )
    )


    def prepare_perturbed_dataset(

        raw_dataframe

    ):

        model_dataframe = (
            raw_dataframe.copy()
        )


        model_dataframe[
            "text"
        ] = (

            model_dataframe[
                "text"
            ]

            .apply(
                model_preprocess
            )
        )


        dataset = (

            Dataset

            .from_pandas(

                model_dataframe,

                preserve_index=False
            )

            .map(

                tokenize_function,

                batched=True
            )
        )


        return dataset


    for dimension in DIMENSIONS:


        print(
            "\n\n"
            +
            "=" * 100
        )


        print(
            MODEL_LABEL,
            "-",
            dimension
        )


        print(
            "=" * 100
        )


        for severity in (
            SEVERITIES
        ):


            print(

                "\nSeverity:",

                int(
                    severity
                    *
                    100
                ),

                "%"
            )


            for perturb_seed in (
                PERTURBATION_SEEDS
            ):


                key = (

                    dimension,

                    severity,

                    perturb_seed
                )


                raw_perturbed_df = (

                    PERTURBED_TEST_SETS[
                        key
                    ]
                )


                perturbed_dataset = (

                    prepare_perturbed_dataset(
                        raw_perturbed_df
                    )
                )


                perturbed_result = (

                    evaluate_dataset(

                        trainer,

                        perturbed_dataset
                    )
                )


                perturbed_accuracy = (

                    perturbed_result[
                        "accuracy"
                    ]
                )


                perturbed_macro_f1 = (

                    perturbed_result[
                        "macro_f1"
                    ]
                )


                perturbed_weighted_f1 = (

                    perturbed_result[
                        "weighted_f1"
                    ]
                )


                delta_f1 = (

                    clean_macro_f1

                    -

                    perturbed_macro_f1
                )


                diagnostic = (

                    diagnostics_df[
                        (
                            diagnostics_df[
                                "Dimension"
                            ]
                            ==
                            dimension
                        )

                        &

                        (
                            diagnostics_df[
                                "Severity"
                            ]
                            ==
                            severity
                        )

                        &

                        (
                            diagnostics_df[
                                "Seed"
                            ]
                            ==
                            perturb_seed
                        )
                    ]

                    .iloc[0]
                )


                ALL_RESULTS.append({

                    "Model":
                        MODEL_LABEL,

                    "Dimension":
                        dimension,

                    "Severity":
                        severity,

                    "Severity_Percent":
                        int(
                            severity
                            *
                            100
                        ),

                    "Perturbation_Seed":
                        perturb_seed,

                    "Clean_Accuracy":
                        clean_accuracy,

                    "Clean_Macro_F1":
                        clean_macro_f1,

                    "Clean_Weighted_F1":
                        clean_weighted_f1,

                    "Perturbed_Accuracy":
                        perturbed_accuracy,

                    "Perturbed_Macro_F1":
                        perturbed_macro_f1,

                    "Perturbed_Weighted_F1":
                        perturbed_weighted_f1,

                    "Delta_F1":
                        delta_f1,

                    "Realized_Rate":
                        diagnostic[
                            "Realized_Rate"
                        ],

                    "Changed_Instance_Rate":
                        diagnostic[
                            "Changed_Instance_Rate"
                        ]
                })


                print(

                    f"Seed {perturb_seed}"

                    f" | Macro-F1="
                    f"{perturbed_macro_f1 * 100:.2f}"

                    f" | Delta="
                    f"{delta_f1 * 100:.2f}"

                    f" | Realized="
                    f"{diagnostic['Realized_Rate'] * 100:.2f}%"

                    f" | Changed texts="
                    f"{diagnostic['Changed_Instance_Rate'] * 100:.2f}%"
                )


                del perturbed_dataset

                gc.collect()


    del trainer

    del model

    del tokenizer

    del train_dataset

    del clean_test_dataset


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    print(
        "\nFinished:",
        MODEL_LABEL
    )


results_df = pd.DataFrame(
    ALL_RESULTS
)


results_df.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "all_individual_runs.csv"
    ),

    index=False
)


summary_df = (

    results_df

    .groupby(

        [
            "Model",
            "Dimension",
            "Severity_Percent"
        ],

        as_index=False
    )

    .agg(

        Clean_Accuracy=(

            "Clean_Accuracy",

            "first"
        ),

        Clean_Macro_F1=(

            "Clean_Macro_F1",

            "first"
        ),

        Clean_Weighted_F1=(

            "Clean_Weighted_F1",

            "first"
        ),

        Accuracy_Mean=(

            "Perturbed_Accuracy",

            "mean"
        ),

        Accuracy_SD=(

            "Perturbed_Accuracy",

            "std"
        ),

        Macro_F1_Mean=(

            "Perturbed_Macro_F1",

            "mean"
        ),

        Macro_F1_SD=(

            "Perturbed_Macro_F1",

            "std"
        ),

        Weighted_F1_Mean=(

            "Perturbed_Weighted_F1",

            "mean"
        ),

        Weighted_F1_SD=(

            "Perturbed_Weighted_F1",

            "std"
        ),

        Delta_F1_Mean=(

            "Delta_F1",

            "mean"
        ),

        Delta_F1_SD=(

            "Delta_F1",

            "std"
        ),

        Realized_Rate_Mean=(

            "Realized_Rate",

            "mean"
        ),

        Realized_Rate_SD=(

            "Realized_Rate",

            "std"
        ),

        Changed_Instance_Rate_Mean=(

            "Changed_Instance_Rate",

            "mean"
        ),

        N_Runs=(

            "Perturbation_Seed",

            "count"
        )
    )
)


summary_100 = (
    summary_df.copy()
)


columns_to_scale = [

    "Clean_Accuracy",

    "Clean_Macro_F1",

    "Clean_Weighted_F1",

    "Accuracy_Mean",

    "Accuracy_SD",

    "Macro_F1_Mean",

    "Macro_F1_SD",

    "Weighted_F1_Mean",

    "Weighted_F1_SD",

    "Delta_F1_Mean",

    "Delta_F1_SD",

    "Realized_Rate_Mean",

    "Realized_Rate_SD",

    "Changed_Instance_Rate_Mean"
]


for column in (
    columns_to_scale
):

    summary_100[
        column
    ] *= 100


clean_results_df = (
    pd.DataFrame(
        CLEAN_RESULTS
    )
)


for column in [

    "Clean_Accuracy",

    "Clean_Macro_F1",

    "Clean_Weighted_F1"

]:

    clean_results_df[
        column
    ] *= 100


print(
    "\n\n"
    +
    "=" * 100
)


print(
    "CLEAN SENTIMENT RESULTS"
)


print(
    "=" * 100
)


display(

    clean_results_df

    .round(3)
)


print(
    "\n\n"
    +
    "#" * 120
)


print(
    "FINAL SENTIMENT ROBUSTNESS RESULTS"
)


print(
    "#" * 120
)


display(

    summary_100[
        [
            "Model",

            "Dimension",

            "Severity_Percent",

            "Clean_Macro_F1",

            "Macro_F1_Mean",

            "Macro_F1_SD",

            "Delta_F1_Mean",

            "Delta_F1_SD",

            "Accuracy_Mean",

            "Accuracy_SD",

            "Realized_Rate_Mean",

            "Changed_Instance_Rate_Mean",

            "N_Runs"
        ]
    ]

    .round(3)
)


paper_df = (
    summary_100.copy()
)


paper_df[
    "Macro-F1"
] = (

    paper_df[
        "Macro_F1_Mean"
    ]

    .map(
        lambda x:
            f"{x:.2f}"
    )

    +

    " ± "

    +

    paper_df[
        "Macro_F1_SD"
    ]

    .map(
        lambda x:
            f"{x:.2f}"
    )
)


paper_df[
    "Delta-F1"
] = (

    paper_df[
        "Delta_F1_Mean"
    ]

    .map(
        lambda x:
            f"{x:.2f}"
    )

    +

    " ± "

    +

    paper_df[
        "Delta_F1_SD"
    ]

    .map(
        lambda x:
            f"{x:.2f}"
    )
)


paper_df[
    "Accuracy"
] = (

    paper_df[
        "Accuracy_Mean"
    ]

    .map(
        lambda x:
            f"{x:.2f}"
    )

    +

    " ± "

    +

    paper_df[
        "Accuracy_SD"
    ]

    .map(
        lambda x:
            f"{x:.2f}"
    )
)


paper_df[
    "Realized Severity"
] = (

    paper_df[
        "Realized_Rate_Mean"
    ]

    .map(
        lambda x:
            f"{x:.2f}%"
    )
)


paper_table = (

    paper_df[
        [
            "Model",

            "Dimension",

            "Severity_Percent",

            "Clean_Macro_F1",

            "Macro-F1",

            "Accuracy",

            "Delta-F1",

            "Realized Severity"
        ]
    ]

    .sort_values(
        [
            "Model",
            "Dimension",
            "Severity_Percent"
        ]
    )
)


print(
    "\n\n"
    +
    "=" * 110
)


print(
    "PAPER-READY SENTIMENT TABLE"
)


print(
    "=" * 110
)


display(
    paper_table
)


dimension_ranking = (

    summary_100

    .groupby(
        [
            "Model",
            "Dimension"
        ],

        as_index=False
    )

    .agg(

        Mean_Delta_F1=(

            "Delta_F1_Mean",

            "mean"
        )
    )

    .sort_values(
        [
            "Model",
            "Mean_Delta_F1"
        ],

        ascending=[
            True,
            False
        ]
    )
)


print(
    "\n\n"
    +
    "=" * 100
)


print(
    "QUALITY-DIMENSION SENSITIVITY RANKING"
)


print(
    "=" * 100
)


display(

    dimension_ranking

    .round(3)
)


cross_model_summary = (

    summary_100

    .groupby(
        [
            "Dimension",
            "Severity_Percent"
        ],

        as_index=False
    )

    .agg(

        Mean_Macro_F1=(

            "Macro_F1_Mean",

            "mean"
        ),

        Mean_Delta_F1=(

            "Delta_F1_Mean",

            "mean"
        )
    )
)


print(
    "\n\n"
    +
    "=" * 100
)


print(
    "CROSS-MODEL SENTIMENT SUMMARY"
)


print(
    "=" * 100
)


display(

    cross_model_summary

    .round(3)
)


severity_30 = (

    summary_100[
        summary_100[
            "Severity_Percent"
        ]
        ==
        30
    ]

    [
        [
            "Model",

            "Dimension",

            "Macro_F1_Mean",

            "Delta_F1_Mean"
        ]
    ]

    .sort_values(
        [
            "Model",
            "Delta_F1_Mean"
        ],

        ascending=[
            True,
            False
        ]
    )
)


print(
    "\n\n"
    +
    "=" * 100
)


print(
    "SENSITIVITY AT 30% SEVERITY"
)


print(
    "=" * 100
)


display(

    severity_30

    .round(3)
)


print(
    "\n\n"
    +
    "=" * 100
)


print(
    "SEVERITY MONOTONICITY CHECK"
)


print(
    "=" * 100
)


for MODEL_LABEL in (
    MODELS.keys()
):

    for dimension in (
        DIMENSIONS
    ):

        subset = (

            summary_100[
                (
                    summary_100[
                        "Model"
                    ]
                    ==
                    MODEL_LABEL
                )

                &

                (
                    summary_100[
                        "Dimension"
                    ]
                    ==
                    dimension
                )
            ]

            .sort_values(
                "Severity_Percent"
            )
        )


        deltas = (

            subset[
                "Delta_F1_Mean"
            ]

            .values
        )


        monotonic = all(

            deltas[idx]
            <=
            deltas[idx + 1]

            for idx
            in range(
                len(deltas) - 1
            )
        )


        print(

            f"{MODEL_LABEL:18s}"

            f" | {dimension:20s}"

            f" | Monotonic = {monotonic}"
        )


summary_100.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "five_dimensions_summary.csv"
    ),

    index=False
)


paper_table.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "paper_ready_sentiment_results.csv"
    ),

    index=False
)


dimension_ranking.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "dimension_sensitivity_ranking.csv"
    ),

    index=False
)


cross_model_summary.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "cross_model_summary.csv"
    ),

    index=False
)


severity_30.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "severity_30_ranking.csv"
    ),

    index=False
)


clean_results_df.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "clean_model_results.csv"
    ),

    index=False
)


diagnostics_df.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "all_perturbation_diagnostics.csv"
    ),

    index=False
)


diagnostic_display.to_csv(

    os.path.join(

        OUTPUT_DIR,

        "perturbation_summary.csv"
    ),

    index=False
)


print(
    "\n\n"
    +
    "#" * 100
)


print(
    "ArSAS SENTIMENT EXPERIMENT COMPLETED"
)


print(
    "#" * 100
)


print(
    "\nTask:"
)

print(
    "Sentiment Analysis"
)


print(
    "\nInput:"
)

print(
    "Tweet_text"
)


print(
    "\nTarget:"
)

print(
    "Sentiment_label"
)


print(
    "\nModels:"
)

print(
    "1. AraBERTv2"
)

print(
    "2. CAMeLBERT-Mix"
)

print(
    "3. XLM-R"
)


print(
    "\nPrimary result file:"
)


print(

    os.path.join(

        OUTPUT_DIR,

        "paper_ready_sentiment_results.csv"
    )
)


print(
    "\nFINAL PAPER-READY RESULTS:"
)


display(
    paper_table
)
